In [1]:
import requests
from bs4 import BeautifulSoup
import json
import os
import socket
import subprocess
import time

def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("", 0))
        return s.getsockname()[1]

def wait_for_server(port, timeout=30):
    start = time.time()

    while time.time() - start < timeout:
        try:
            requests.get(f"http://localhost:{port}", timeout=1)
            return
        except requests.ConnectionError:
            time.sleep(0.5)

    raise TimeoutError(f"Server did not start on port {port}")

PORT = find_free_port()
pwd = os.path.dirname(os.getcwd())

server = subprocess.Popen(
    ["npm", "run", "dev", "--", "--port", str(PORT)],
    cwd=pwd,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

try:
    wait_for_server(PORT)
    print(f"Server running on port {PORT}")

    searchable_pages = []

    folders = ["sta", "sol", "dyn", "thermodynamics", "md", "mf"]

    for folder in folders:
        dir_path = os.path.join(pwd, "src/pages", folder)

        for page in os.listdir(dir_path):
            if page.endswith(".astro"):
                searchable_pages.append(
                    os.path.join(folder, page.replace(".astro", ""))
                )

    page_text = []

    for page in searchable_pages:
        print(page)

        url = f"http://localhost:{PORT}/{page}"
        html = requests.get(url).text

        soup = BeautifulSoup(html, features="html.parser")

        for tag in soup(["script", "style", "nav", "header", "footer"]):
            tag.extract()

        for el_id in ["nav_container", "page_title", "title_bar"]:
            el = soup.find(attrs={"id": el_id})
            if el:
                el.extract()

        text = soup.get_text()

        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = " ".join(chunk for chunk in chunks if chunk)

        banned_strings = [
            "Derivation +",
            "Solution +",
            "All courses Statics Dynamics Solid Mechanics ",
            "Scroll back to top"
        ]

        for b in banned_strings:
            text = text.replace(b, "")

        page_text.append({
            "title": page.split("/")[1].replace("_", " ").capitalize(),
            "text": text,
            "link": "/" + page,
            "course": page.split("/")[0]
        })

    output_path = os.path.join(pwd, "src/search.json")

    with open(output_path, "w") as f:
        json.dump(page_text, f, indent=2)

finally:
    server.terminate()
    print("Server stopped")

Server running on port 54747
sta/hydrostatic_fluid_pressure
sta/free_body_diagrams
sta/cartesian_coordinates
sta/friction
sta/moments
sta/virtual_work
sta/track_and_field_starting_blocks
sta/reaction_forces
sta/frames_and_machines
sta/geometric_properties
sta/internal_forces
sta/force_systems
sta/introduction
sta/pltips
sta/vectors_scalars
sta/trusses
sol/superposition
sol/beam_deflection
sol/sign_conventions
sol/axial_loading
sol/failure_theories
sol/units
sol/stress
sol/combined_loading
sol/stress_transformation
sol/torsion
sol/material_properties
sol/geometric_properties
sol/transverse_shear
sol/shear_&_moment_diagrams
sol/strain
sol/design_considerations
sol/buckling
sol/thermal_loading
sol/bending
sol/pressure_vessels
sol/sympy_tutorial
sol/intro
sol/overview
dyn/celestial_velocities
dyn/banked_turns
dyn/multi_body_systems
dyn/coordinate_systems
dyn/shortest_flight_paths
dyn/rigid_body_kinematics
dyn/steering_geometry
dyn/contact_and_rolling
dyn/rigid_body_kinetics
dyn/vectors
dyn